# Phase 3 — Your First Vector Database with Qdrant

**Goal:** learn how to create a collection, store text chunks and embeddings, attach metadata, embed a query, run top-K similarity search, and inspect the results.

This notebook uses:
- **Qdrant** — a vector database with metadata payloads and local persistence.
- **Sentence Transformers** — an open-source embedding model that runs locally.
- **A small sample knowledge base** — no API key or paid model required.

> **Important:** Qdrant returns a cosine **similarity score** for this collection. Higher scores indicate closer matches; scores are not calibrated confidence percentages.


## 0. Install the packages with `uv`

```bash
uv add qdrant-client sentence-transformers jupyter
```


## 1. Import libraries and load a local embedding model

An **embedding** is a list of numbers representing the meaning of text. Texts with similar meanings tend to have vectors that are close together.

We use `all-MiniLM-L6-v2`, a compact Sentence Transformers model. It is free to use locally; the first download requires internet access.


In [1]:
from pathlib import Path

from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2


## 2. Prepare example document chunks

In a real RAG pipeline, you would extract text from PDFs, Word files, or web pages and split it into chunks. Here, each string is already a chunk.

Each chunk has:
- a unique **ID**
- the chunk **text**
- **metadata** such as filename, page, and section

Keep chunks small enough to retrieve a focused passage, but large enough to preserve context.

In [2]:
chunks = [
    {
        "id": "hr-001",
        "text": "Employees can request annual leave through the HR portal. Their manager must approve the request before the leave is confirmed.",
        "metadata": {"filename": "employee_handbook.pdf", "page": 4, "section": "Annual Leave"},
    },
    {
        "id": "hr-002",
        "text": "Unused annual leave is handled according to the company's leave policy and applicable local employment rules.",
        "metadata": {"filename": "employee_handbook.pdf", "page": 5, "section": "Leave Balance"},
    },
    {
        "id": "it-001",
        "text": "To reset a forgotten password, open the company identity portal and choose Forgot password. Complete the verification steps.",
        "metadata": {"filename": "it_support_guide.pdf", "page": 2, "section": "Password Reset"},
    },
    {
        "id": "it-002",
        "text": "Multi-factor authentication adds a second verification step, such as an authenticator app prompt, when signing in.",
        "metadata": {"filename": "it_security_guide.pdf", "page": 3, "section": "Multi-Factor Authentication"},
    },
    {
        "id": "dev-001",
        "text": "A vector database stores embedding vectors and can retrieve records whose vectors are close to a query vector.",
        "metadata": {"filename": "rag_basics.pdf", "page": 1, "section": "Vector Databases"},
    },
    {
        "id": "dev-002",
        "text": "In retrieval-augmented generation, the application retrieves relevant document chunks and provides them as context to a language model.",
        "metadata": {"filename": "rag_basics.pdf", "page": 2, "section": "RAG Workflow"},
    },
    {
        "id": "dev-003",
        "text": "Chunk metadata can identify the source filename, page number, section title, and other filtering fields.",
        "metadata": {"filename": "rag_basics.pdf", "page": 3, "section": "Metadata"},
    },
    {
        "id": "dev-004",
        "text": "Random access memory (RAM) is a type of computer memory that can be accessed randomly, allowing for fast data retrieval.",
        "metadata": {"filename": "cpu_basics.pdf", "page": 1, "section": "Metadata"},
    },
]

texts = [item["text"] for item in chunks]
ids = [item["id"] for item in chunks]
metadatas = [item["metadata"] for item in chunks]

print(f"Chunks: {len(chunks)}")
print("Example metadata:", metadatas[0])

Chunks: 8
Example metadata: {'filename': 'employee_handbook.pdf', 'page': 4, 'section': 'Annual Leave'}


## 3. Create a Qdrant collection (the vector index)

A **collection** is similar to a table or index that holds points, vectors, and payloads (metadata plus any text you want to store).

This notebook uses a persistent local folder named `qdrant_db`. Your data can remain there between notebook sessions. We delete only this tutorial collection first so rerunning the notebook is predictable.


In [3]:
DB_PATH = str(Path("qdrant_db").resolve())
COLLECTION_NAME = "phase3_learning"

#client = QdrantClient(path=DB_PATH)
client = QdrantClient(url="http://localhost:6333")

# Reset only this tutorial collection so rerunning the notebook is predictable.
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

# all-MiniLM-L6-v2 produces 384-dimensional vectors.
VECTOR_SIZE = model.get_sentence_embedding_dimension()

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=VECTOR_SIZE,
        distance=models.Distance.COSINE,
    ),
)

print("Database folder:", DB_PATH)
print("Collection:", COLLECTION_NAME)
print("Vector size:", VECTOR_SIZE)


C:\Users\abdul\AppData\Local\Temp\ipykernel_24436\1132299081.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  VECTOR_SIZE = model.get_sentence_embedding_dimension()


Database folder: D:\rahman3593\AI\RAG\notebooks\qdrant_db
Collection: phase3_learning
Vector size: 384


## 4. Create embeddings and add chunks, IDs, and metadata

We create one embedding per chunk. `model.encode()` returns numeric vectors. Then we store each vector in Qdrant as a **point**. A point has an ID, a vector, and a payload.

Qdrant point IDs must be unsigned integers or UUIDs, so we use numeric point IDs and keep the original readable chunk ID (such as `hr-001`) in the payload as `chunk_id`.


In [4]:
embeddings = model.encode(texts, normalize_embeddings=True).tolist()

print("Number of embeddings:", len(embeddings))
print("Dimensions of one embedding:", len(embeddings[0]))
print("First 8 values:", embeddings[0][:8])

points = [
    models.PointStruct(
        id=index + 1,
        vector=embedding,
        payload={
            "chunk_id": ids[index],
            "text": texts[index],
            **metadatas[index],
        },
    )
    for index, embedding in enumerate(embeddings)
]

client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True,
)

print("Points stored:", client.count(collection_name=COLLECTION_NAME, exact=True).count)


Number of embeddings: 8
Dimensions of one embedding: 384
First 8 values: [0.012330853380262852, 0.05176148563623428, 0.005960152484476566, 0.03837430104613304, 0.08637141436338425, 0.036474090069532394, -0.055599652230739594, -0.10203832387924194]
Points stored: 8


## 5. Embed a question and run top-K similarity search

A query must be embedded using the **same embedding model** as the stored chunks. We pass its vector to Qdrant and ask for the `K` nearest points.

Try changing the question to:
- `How can I change my password?`
- `What does a vector database do?`
- `How does RAG use retrieved documents?`


In [5]:
query_text = "what is RAM?"

query_embedding = model.encode(
    [query_text],
    normalize_embeddings=True,
).tolist()[0]

print("Question:", query_text)
print("Query vector dimensions:", len(query_embedding))


Question: what is RAM?
Query vector dimensions: 384


## 6. Inspect returned text, scores, and metadata

Qdrant returns a list of scored points. Each point has a payload containing the original text and metadata.

The `score` is **cosine similarity** for this collection:
- scores closer to `1` generally mean more similar
- lower scores mean less similar
- it is **not** a percentage or a calibrated confidence score


In [7]:
def search_and_print(question: str, k: int = 3):
    q_embedding = model.encode(
        [question],
        normalize_embeddings=True,
    ).tolist()[0]

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embedding,
        limit=min(k, client.count(collection_name=COLLECTION_NAME, exact=True).count),
        with_payload=True,
        with_vectors=False,
    ).points

    print(f"\\nQuestion: {question!r} | K={k}")
    print("=" * 90)

    for rank, point in enumerate(results, start=1):
        payload = point.payload or {}
        print(
            f"Rank: {rank} | ID: {payload.get('chunk_id')} | "
            f"cosine similarity: {point.score:.4f}"
        )
        print(
            f"Source: {payload.get('filename')} | Page: {payload.get('page')} | "
            f"Section: {payload.get('section')}"
        )
        print(f"Text: {payload.get('text')}")
        print("-" * 90)

    return results

results = search_and_print(query_text, k=1)


\nQuestion: 'what is RAM?' | K=1
Rank: 1 | ID: dev-004 | cosine similarity: 0.7236
Source: cpu_basics.pdf | Page: 1 | Section: Metadata
Text: Random access memory (RAM) is a type of computer memory that can be accessed randomly, allowing for fast data retrieval.
------------------------------------------------------------------------------------------


## 7. Experiment with K = 1, 3, and 5

`K` controls how many candidate chunks are returned. It does **not** change the embeddings or the stored index.

In [8]:
for k in [1, 3, 5]:
    search_and_print(query_text, k=k)

\nQuestion: 'what is RAM?' | K=1
Rank: 1 | ID: dev-004 | cosine similarity: 0.7236
Source: cpu_basics.pdf | Page: 1 | Section: Metadata
Text: Random access memory (RAM) is a type of computer memory that can be accessed randomly, allowing for fast data retrieval.
------------------------------------------------------------------------------------------
\nQuestion: 'what is RAM?' | K=3
Rank: 1 | ID: dev-004 | cosine similarity: 0.7236
Source: cpu_basics.pdf | Page: 1 | Section: Metadata
Text: Random access memory (RAM) is a type of computer memory that can be accessed randomly, allowing for fast data retrieval.
------------------------------------------------------------------------------------------
Rank: 2 | ID: dev-001 | cosine similarity: 0.2352
Source: rag_basics.pdf | Page: 1 | Section: Vector Databases
Text: A vector database stores embedding vectors and can retrieve records whose vectors are close to a query vector.
----------------------------------------------------------------

### What should you observe?

- **K = 1:** only the closest chunk is returned. This can be precise, but may omit useful context.
- **K = 3:** gives a few candidates and is a useful starting point for many small RAG experiments.
- **K = 5:** returns more context, but some results may be less relevant.
- Inspect the source metadata to see which file, page, and section supplied each result.
- Do not assume every returned chunk is relevant just because it appears in the top K.

## 8. Filter by metadata

Metadata lets you restrict retrieval to a source or section. For example, this query only searches chunks from `rag_basics.pdf`.

In Qdrant, metadata is stored in the point's **payload**. A payload filter restricts which points are considered before ranking.


In [ ]:
question = "What is a vector database?"
q_embedding = model.encode([question], normalize_embeddings=True).tolist()[0]

filtered_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=q_embedding,
    limit=3,
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="filename",
                match=models.MatchValue(value="rag_basics.pdf"),
            )
        ]
    ),
    with_payload=True,
    with_vectors=False,
).points

for rank, point in enumerate(filtered_results, start=1):
    payload = point.payload or {}
    print(
        f"{rank}. {payload.get('chunk_id')} | "
        f"similarity={point.score:.4f} | "
        f"filename={payload.get('filename')} | page={payload.get('page')} | "
        f"section={payload.get('section')}"
    )
    print(payload.get("text"), "\\n")


## 9. Persistence: reopen the collection

Because we used `QdrantClient(path=DB_PATH)`, Qdrant stores its local data in the `qdrant_db` folder.

Close the client before opening the same local database path again. If you restart the kernel, run the imports/model setup first, then this cell to verify persistence.


In [ ]:
client.close()

client_again = QdrantClient(path=DB_PATH)
print(
    "Records after reopening:",
    client_again.count(collection_name=COLLECTION_NAME, exact=True).count,
)
print("Collection name:", COLLECTION_NAME)


## 10. Mini exercises

Try these yourself before looking at the suggested answers.

1. Change the query to `Who approves annual leave?` and compare K=1, 3, and 5.
2. Add one new chunk with a unique ID, text, and metadata; embed it and store it with `client.upsert(...)`.
3. Search only within `it_support_guide.pdf` using a Qdrant payload filter.
4. Print just the IDs and similarity scores in a compact list.
5. Explain in your own words the difference between **embedding**, **vector database**, **payload/metadata**, **top-K**, and **similarity score**.

### Example: add one more chunk

Uncomment and run this cell after adapting the content. The numeric Qdrant point ID must be unique. The readable chunk ID is stored in the payload.


In [ ]:
# extra_text = "Employees should contact the IT help desk if account verification fails."
# extra_embedding = model.encode([extra_text], normalize_embeddings=True).tolist()[0]
#
# client.upsert(
#     collection_name=COLLECTION_NAME,
#     points=[
#         models.PointStruct(
#             id=100,  # Use a unique numeric point ID
#             vector=extra_embedding,
#             payload={
#                 "chunk_id": "it-003",
#                 "text": extra_text,
#                 "filename": "it_support_guide.pdf",
#                 "page": 4,
#                 "section": "Account Help",
#             },
#         )
#     ],
#     wait=True,
# )
#
# print(
#     "Updated record count:",
#     client.count(collection_name=COLLECTION_NAME, exact=True).count,
# )


## Key takeaways

- **Embedding model:** converts text into vectors.
- **Vector database:** stores vectors and supports nearest-neighbor retrieval.
- **Point ID:** unique identifier required by Qdrant for each stored point.
- **Payload/metadata:** structured source details useful for filtering and citations; it can also hold the text chunk.
- **Top-K:** number of nearest results to return.
- **Cosine similarity:** a ranking score here; higher is closer.
- **Persistence:** Qdrant's local `path` stores data under `qdrant_db`.
- **RAG connection:** retrieve useful chunks first, then pass their text to a language model as context.

**Next phase:** load a real PDF, split it into chunks, store filename/page/section payload metadata, and use the retrieved passages as context in a simple RAG question-answering pipeline.
